# Environment check

Run this on a Colab or Kaggle GPU runtime (Runtime › Change runtime type › GPU).

In [ ]:
import sys, torch
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# A quick matrix multiply on the GPU to confirm it works
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(4096, 4096, device=device)
t = time.time(); y = x @ x
if device == 'cuda': torch.cuda.synchronize()
print(f'{device}: 4096x4096 matmul in {time.time() - t:.3f}s')

## Mount Google Drive (Colab only)

Checkpoints and datasets live in Drive so they survive disconnects. The folder is created the first time this runs; after that the cell just reconnects. Copy this cell to the top of any notebook that trains a model.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/ai-engineering-journey'
for sub in ['checkpoints', 'data', 'outputs']:
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)  # no-op if it already exists
print('Saving to', OUT, '->', sorted(os.listdir(OUT)))

## Optional: call an LLM from Colab

Pick a provider: `openrouter` (one key for Claude, GPT, Llama and more), `together` (open models, also fine-tuning) or `anthropic`. The cell asks for that provider's key each session and keeps it only in this runtime's memory. Nothing is saved to the notebook, GitHub or Colab secrets. Model ids follow each provider's own model list.

In [ ]:
%pip install -q openai anthropic
import os, getpass
PROVIDER = 'openrouter'  # or 'together' or 'anthropic'
CONFIG = {
    'openrouter': ('OPENROUTER_API_KEY', 'https://openrouter.ai/api/v1', 'meta-llama/llama-3.3-70b-instruct'),
    'together': ('TOGETHER_API_KEY', 'https://api.together.xyz/v1', 'meta-llama/Llama-3.3-70B-Instruct-Turbo'),
    'anthropic': ('ANTHROPIC_API_KEY', None, 'claude-opus-5-5'),
}
env_var, base_url, model = CONFIG[PROVIDER]
key = os.environ.get(env_var) or getpass.getpass(f'{env_var} (hidden): ')
prompt = 'In one sentence, what is a KV cache?'

if PROVIDER == 'anthropic':
    import anthropic
    r = anthropic.Anthropic(api_key=key).messages.create(model=model, max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}])
    print(''.join(b.text for b in r.content if b.type == 'text'))
else:
    from openai import OpenAI
    r = OpenAI(api_key=key, base_url=base_url).chat.completions.create(model=model, max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}])
    print(r.choices[0].message.content)